<a href="https://colab.research.google.com/github/Samir-atra/Private-ML/blob/main/MulticlassLR/Dense_noise_leaf_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook serves as a template for a neural network implemented with NumPy.

In [1]:
# Import necessary libraries
import tensorflow as tf
import numpy as np
import os
import PIL.Image
from PIL import ImageOps
import PIL
import pathlib
import matplotlib.pyplot as plt
import datetime
import IPython
import sklearn
import cv2
import sys

# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# Define file paths for the datasets
data_path = pathlib.Path('/content/drive/MyDrive/LeafDataset/Leaves/Leafdataset/Training/')
data_path_test = pathlib.Path('/content/drive/MyDrive/LeafDataset/Leaves/Leafdataset/Testing/')

# Create the training dataset from the directory
dataset_path = tf.keras.utils.image_dataset_from_directory(
    data_path,
    labels='inferred',
    validation_split=0.2,
    subset='training',
    seed=1,
    batch_size=5,
    image_size=(180, 180),
    color_mode="rgb",
    shuffle=True
)

# Create the validation dataset from the directory
dataset_path_val = tf.keras.utils.image_dataset_from_directory(
    data_path,
    labels='inferred',
    validation_split=0.2,
    subset='validation',
    seed=2,
    batch_size=5,
    image_size=(180, 180),
    color_mode="rgb",
    shuffle=True
)

# Create the test dataset from the directory
dataset_path_test = tf.keras.utils.image_dataset_from_directory(
    data_path_test,
    labels='inferred',
    seed=3,
    batch_size=5,
    image_size=(180, 180),
    color_mode="rgb",
    shuffle=True
)

In [3]:
# Optimize dataset performance by caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE
dataset_path = dataset_path.cache().prefetch(buffer_size=AUTOTUNE)
dataset_path_val = dataset_path_val.cache().prefetch(buffer_size=AUTOTUNE)

In [4]:
# Define the number of classes for classification
num_classes = 2

# Create a sequential model
model = tf.keras.Sequential([
    # Resize images to 60x60
    tf.keras.layers.Resizing(60, 60),
    # Rescale pixel values to [0, 1]
    tf.keras.layers.Rescaling(1./255),
    # Flatten the images to a 1D array
    tf.keras.layers.Flatten(),
    # Add dense layers with ReLU activation and L2 regularization
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.2),
    # Output layer with sigmoid activation for binary classification
    tf.keras.layers.Dense(num_classes, activation='sigmoid')
])

# Compile the model with Adam optimizer and sparse categorical crossentropy loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Train the model for 10 epochs
model.fit(
    dataset_path,
    epochs=10,
    validation_data=dataset_path_val
)

# Evaluate the model on the test dataset
model.evaluate(dataset_path_test, batch_size=5, verbose=2)

# Print the model summary
model.summary()